# Action Recognition
 Foundations of Machine Learning SS26    
 Group 11 - G. Sammet, M. Schlichting
### Overall Goal
- Classify 4 actions
### Dataset 
- RGB videos
    - ca 2 seconds long
    - resolution 1920x1080
- ca. 900 videos per action
- 4 actions: waving, capitulate, cross arms and clapping

    **Aquisition:**    
    We contacted ROSE Lab to get permission for the download. Due to hardware limits we downloaded the zip files in batched and ran the ``process_NTU.py`` to extract only the action classes we wanted.

    *The research in this project used the NTU RGB+D (or NTU RGB+D 120) Action Recognition Dataset made available by the ROSE Lab at the Nanyang Technological University, Singapore.*

    ROSE Lab. Action recognition datasets: “NTU RGB+D” dataset and “NTU RGB+D 120” dataset [Dataset]. https://rose1.ntu.edu.sg/dataset/actionRecognition/  

### Imports

In [1]:
from pathlib import Path
import random
import shutil

import torch
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## Experiment 1: Simple One-Frame-Model
single frame -> pre-trained ResNet18 -> custom classifier output layer

### Load ResNet18 Model & Dataset

In [5]:
# load pretrained model
weights = ResNet18_Weights.DEFAULT
resnet_model = resnet18(weights=weights)

# modify last layer to 4 classes
resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, 4)

# define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet_model.fc.parameters(), lr=0.001, momentum=0.9)

# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
# load data and transform
output_dir = Path("data/three_quarter_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_three_quarter = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_three_quarter = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_three_quarter = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Training

In [7]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # Print the results for the current epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [ ]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model_three_quarter = resnet_model.to(device)

train(model_three_quarter, train_loader_three_quarter, val_loader_three_quarter, criterion, optimizer, num_epochs=10)

# save checkpoint of model
checkpoint = {
    "model_three_quarter_state_dict": model_three_quarter.state_dict(),
    "optimizer_model_three_quarter_state_dict": optimizer.state_dict(),
    "epoch": 10,
}

torch.save(
    checkpoint,
    "model_three_quarter_checkpoint.pth"
)

Epoch [1/10], train loss: 1.1948, train acc: 0.4292, val loss: 0.9846, val acc: 0.5491    
Epoch [2/10], train loss: 0.9416, train acc: 0.5825, val loss: 0.8774, val acc: 0.5947    
Epoch [3/10], train loss: 0.8520, train acc: 0.6257, val loss: 0.8262, val acc: 0.6316    
Epoch [4/10], train loss: 0.8003, train acc: 0.6652, val loss: 0.7976, val acc: 0.6474    
Epoch [5/10], train loss: 0.7734, train acc: 0.6678, val loss: 0.7642, val acc: 0.6579    
Epoch [6/10], train loss: 0.7405, train acc: 0.6885, val loss: 0.7513, val acc: 0.6544    
Epoch [7/10], train loss: 0.7162, train acc: 0.6933, val loss: 0.7364, val acc: 0.6667    
Epoch [8/10], train loss: 0.6940, train acc: 0.7073, val loss: 0.7236, val acc: 0.6614    
Epoch [9/10], train loss: 0.6970, train acc: 0.7035, val loss: 0.7247, val acc: 0.6807    
Epoch [10/10], train loss: 0.6785, train acc: 0.7166, val loss: 0.7169, val acc: 0.6789    

# TODO: EVAL FUNCTION

In [9]:
def evaluate_model(model, test_loader, device):
    # Initialize dictionaries to store correct and total predictions
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    # Set the model to evaluation mode
    model.eval()

    # Track the ground truth labels and predictions
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move the inputs and labels to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Collect predictions and labels for metric calculations
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # Update the correct and total predictions
            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    # Calculate accuracy per class
    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname] if total_pred[classname] > 0 else 0
                          for classname in test_loader.dataset.classes}

    # Calculate overall accuracy
    overall_accuracy = accuracy_score(all_labels, all_preds)


    # Print the evaluation results
    print("Accuracy per class:")
    for classname, accuracy in accuracy_per_class.items():
        print(f"{classname}: {accuracy:.4f}")

    print()
    print(f"Overall Accuracy: {overall_accuracy:.4f}")

    

In [ ]:
# load model
checkpoint = torch.load("checkpoints/model_three_quarter_checkpoint.pth",map_location=device)

model_three_quarter.load_state_dict(checkpoint["model_three_quarter_state_dict"])

optimizer.load_state_dict(checkpoint["optimizer_model_three_quarter_state_dict"])
model_three_quarter.to(device)

#evaluate
evaluate_model(model_three_quarter, test_loader_three_quarter, device)

## Experiment 2: 3-Model-3-Frames-Approach
3 frames per video -> one model per frame type -> average over class predictions -> classification

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.SGD(resnet_model.fc.parameters(), lr=0.001, momentum=0.9)

# load datat for middle frame
output_dir = Path("data/middle_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_middle = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_middle = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_middle = DataLoader(test_dataset, batch_size=32, shuffle=False)

# train model 2
model_middle = resnet_model.to(device)
train(model_middle, train_loader_middle, val_loader_middle, criterion, optimizer, num_epochs=10)

# save checkpoint of model
checkpoint = {
    "model_middle_state_dict": model_middle.state_dict(),
    "optimizer_middle_state_dict": optimizer.state_dict(),
    "epoch": 10,
}

torch.save(
    checkpoint,
    "model_middle_checkpoint.pth"
)

Epoch [1/10], train loss: 0.7404, train acc: 0.6712, val loss: 0.7988, val acc: 0.6053    
Epoch [2/10], train loss: 0.7342, train acc: 0.6731, val loss: 0.8046, val acc: 0.6088    
Epoch [3/10], train loss: 0.7204, train acc: 0.6828, val loss: 0.8107, val acc: 0.5930    
Epoch [4/10], train loss: 0.7158, train acc: 0.6840, val loss: 0.7872, val acc: 0.6228    
Epoch [5/10], train loss: 0.7118, train acc: 0.6888, val loss: 0.7908, val acc: 0.6228    
Epoch [6/10], train loss: 0.7216, train acc: 0.6682, val loss: 0.7710, val acc: 0.6351    
Epoch [7/10], train loss: 0.7011, train acc: 0.6855, val loss: 0.7582, val acc: 0.6263    
Epoch [8/10], train loss: 0.6809, train acc: 0.7073, val loss: 0.7867, val acc: 0.6316    
Epoch [9/10], train loss: 0.6856, train acc: 0.7009, val loss: 0.7689, val acc: 0.6228    
Epoch [10/10], train loss: 0.6867, train acc: 0.6979, val loss: 0.8203, val acc: 0.6018

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.SGD(resnet_model.fc.parameters(), lr=0.001, momentum=0.9) 

# load datat for end frame
output_dir = Path("data/end_frame")  
train_dataset = ImageFolder((str(output_dir / "train")), transform=transform)
val_dataset = ImageFolder((str(output_dir / "val")), transform=transform)
test_dataset = ImageFolder((str(output_dir)), transform=transform)

train_loader_end = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_end = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_end = DataLoader(test_dataset, batch_size=32, shuffle=False)

# train model 2
model_end = resnet_model.to(device)
train(model_end, train_loader_end, val_loader_end, criterion, optimizer, num_epochs=10)

# save checkpoint of model
checkpoint = {
    "model_end_state_dict": model_end.state_dict(),
    "optimizer_middle_state_dict": optimizer.state_dict(),
    "epoch": 10,
}

torch.save(
    checkpoint,
    "model_end_checkpoint.pth"
)

Epoch [1/10], train loss: 0.6769, train acc: 0.6930, val loss: 0.7013, val acc: 0.6702    
Epoch [2/10], train loss: 0.6608, train acc: 0.7061, val loss: 0.6932, val acc: 0.6842    
Epoch [3/10], train loss: 0.6517, train acc: 0.7227, val loss: 0.6918, val acc: 0.6544    
Epoch [4/10], train loss: 0.6551, train acc: 0.7166, val loss: 0.7144, val acc: 0.6491    
Epoch [5/10], train loss: 0.6401, train acc: 0.7189, val loss: 0.6965, val acc: 0.6702    
Epoch [6/10], train loss: 0.6340, train acc: 0.7185, val loss: 0.7036, val acc: 0.6667    
Epoch [7/10], train loss: 0.6316, train acc: 0.7275, val loss: 0.6928, val acc: 0.6561    
Epoch [8/10], train loss: 0.6260, train acc: 0.7272, val loss: 0.6936, val acc: 0.6737    
Epoch [9/10], train loss: 0.6219, train acc: 0.7418, val loss: 0.6882, val acc: 0.6754    
Epoch [10/10], train loss: 0.6284, train acc: 0.7275, val loss: 0.6748, val acc: 0.6930    